## 2. Mathematical Derivation

Having established the problem context and our research approach, we now derive the mathematical framework underlying PageRank. This section formalizes the random surfer model as a Markov chain, explains why Power Iteration converges to the steady-state distribution, and introduces the damping factor modification that ensures robustness on real web graphs.

---

### 2.1 The Random Surfer Model

#### 2.1.1 Markov Chain Formulation

PageRank models web navigation as a **discrete-time Markov chain** on a finite state space. We formalize this as follows:

**Definition 2.1 (Discrete-Time Markov Chain):**  
A sequence of random variables $\{X_t\}_{t=0}^{\infty}$ taking values in a finite state space $S = \{1, 2, \ldots, n\}$ is a Markov chain if it satisfies the **Markov property**:

$$
P(X_{t+1} = j \mid X_0 = i_0, X_1 = i_1, \ldots, X_t = i) = P(X_{t+1} = j \mid X_t = i) = M_{ij}
$$

for all states $i, j \in S$ and times $t \geq 0$, where $M_{ij}$ is the **transition probability** from state $i$ to state $j$ [10].

**Interpretation:** The future state $X_{t+1}$ depends only on the current state $X_t$, not on the history of how we arrived at $X_t$. This is the **memoryless property**—past browsing behavior is irrelevant once we know the current page.

**For PageRank:**
- State space $S$ = set of $n$ web pages
- $X_t$ = page visited at time $t$
- $M_{ij}$ = probability of clicking a link from page $j$ to page $i$

The transition matrix $M \in \mathbb{R}^{n \times n}$ encodes all transition probabilities.

#### 2.1.2 Why Finite State Spaces Simplify Probability Theory

**Important clarification:** General Markov chains on continuous or countably infinite state spaces require **measure-theoretic probability** (σ-algebras, Lebesgue integration, conditional expectations as random variables) [11]. However, our web graph has a **finite** number of pages, which dramatically simplifies the mathematics:

- **Probability space:** For finite $S$, the probability space is $(S, 2^S, P)$ where $2^S$ is the power set (all subsets). Every subset is measurable—no need to construct σ-algebras.
- **Probabilities as sums:** $P(X_t = i) = \sum_{j \in S} P(X_{t-1} = j) \cdot M_{ij}$. No integrals or measure theory required.
- **Elementary probability suffices:** All our calculations reduce to matrix-vector operations and finite sums.

This is why we can work with undergraduate linear algebra instead of advanced probability theory—the finite state space makes everything discrete [12].

#### 2.1.3 The Stationary Distribution

**Definition 2.2 (Stationary Distribution):**  
A probability vector $\boldsymbol{\pi} \in \mathbb{R}^n$ is a **stationary distribution** (or steady-state distribution) of the Markov chain if:

$$
\boldsymbol{\pi} = M \boldsymbol{\pi}
$$

and $\sum_{i=1}^{n} \pi_i = 1$, with $\pi_i \geq 0$ for all $i$.

**Physical meaning:** If the random surfer's probability distribution over pages is $\boldsymbol{\pi}$ at time $t$, it remains $\boldsymbol{\pi}$ at time $t+1$. The system is in **equilibrium**.

**Example (3-page cycle from Section 1.3):**

Transition matrix:
$$
M = \begin{bmatrix}
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$

Starting from Page A with certainty: $\mathbf{v}_0 = [1, 0, 0]^T$

- Time step 1: $\mathbf{v}_1 = M\mathbf{v}_0 = [0, 1, 0]^T$ (surfer at Page B)
- Time step 2: $\mathbf{v}_2 = M\mathbf{v}_1 = [0, 0, 1]^T$ (surfer at Page C)
- Time step 3: $\mathbf{v}_3 = M\mathbf{v}_2 = [1, 0, 0]^T$ (back to Page A)

The distribution cycles forever with period 3. However, if we start uniformly: $\mathbf{v}_0 = [\frac{1}{3}, \frac{1}{3}, \frac{1}{3}]^T$, then $M\mathbf{v}_0 = \mathbf{v}_0$ (stationary). This symmetry exists because the graph forms a perfect cycle.

---

### 2.2 Linear Transformation as State Evolution

#### 2.2.1 Matrix-Vector Multiplication as State Update

Each step of the random walk corresponds to a **linear transformation** of the probability distribution:

$$
\mathbf{v}_{t+1} = M \mathbf{v}_t
$$

where $\mathbf{v}_t \in \mathbb{R}^n$ is the probability distribution at time $t$ (with $\|\mathbf{v}_t\|_1 = 1$).

After $k$ steps:
$$
\mathbf{v}_k = M^k \mathbf{v}_0
$$

This is a **composite transformation**—applying the same linear map $M$ repeatedly.

#### 2.2.2 Spectral Decomposition and Long-Term Behavior

To understand what happens as $k \to \infty$, we employ the **spectral decomposition** of $M$ [13].

**Theorem 2.1 (Spectral Decomposition):**  
If $M \in \mathbb{R}^{n \times n}$ is diagonalizable with eigenvalues $\lambda_1, \lambda_2, \ldots, \lambda_n$ and corresponding eigenvectors $\mathbf{q}_1, \mathbf{q}_2, \ldots, \mathbf{q}_n$, then:

$$
M = Q \Lambda Q^{-1}
$$

where $Q = [\mathbf{q}_1 \mid \mathbf{q}_2 \mid \cdots \mid \mathbf{q}_n]$ and $\Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_n)$.

**Consequence for repeated multiplication:**

$$
M^k = Q \Lambda^k Q^{-1} = Q \begin{bmatrix}
\lambda_1^k & & \\
& \lambda_2^k & \\
& & \ddots \\
& & & \lambda_n^k
\end{bmatrix} Q^{-1}
$$

**Decomposing the initial vector:** Any $\mathbf{v}_0$ can be expressed as a linear combination of eigenvectors:

$$
\mathbf{v}_0 = c_1 \mathbf{q}_1 + c_2 \mathbf{q}_2 + \cdots + c_n \mathbf{q}_n
$$

Applying $M^k$:

$$
M^k \mathbf{v}_0 = c_1 \lambda_1^k \mathbf{q}_1 + c_2 \lambda_2^k \mathbf{q}_2 + \cdots + c_n \lambda_n^k \mathbf{q}_n
$$

**Key insight:** If $|\lambda_1| > |\lambda_i|$ for all $i \neq 1$ (the **dominant eigenvalue** is unique), then as $k \to \infty$:

$$
\frac{\lambda_i^k}{\lambda_1^k} \to 0 \quad \text{for all } i \neq 1
$$

Thus:
$$
M^k \mathbf{v}_0 \approx c_1 \lambda_1^k \mathbf{q}_1
$$

The vector converges to a scalar multiple of the **dominant eigenvector** $\mathbf{q}_1$. After normalization, this is our stationary distribution.

**For stochastic matrices:** The Perron-Frobenius theorem (proven in Section 2.3) guarantees that $\lambda_1 = 1$ and $|\lambda_i| < 1$ for $i \neq 1$ (under suitable conditions). Therefore:

$$
M^k \mathbf{v}_0 \to c_1 \mathbf{q}_1 \quad \text{as } k \to \infty
$$

Since $\mathbf{q}_1$ is the eigenvector with $\lambda = 1$, it is the stationary distribution.

---

### 2.3 The Power Iteration Method

#### 2.3.1 Algorithm Description

The **Power Iteration Method** (also called **power method**) is an iterative algorithm for computing the dominant eigenvector of a matrix [14].

**Algorithm 2.1 (Power Iteration for PageRank):**
```
Input: Transition matrix M, tolerance ε > 0, max iterations K
Output: Approximate PageRank vector r

1. Initialize: v = [1/n, 1/n, ..., 1/n]ᵀ  (uniform distribution)
2. For k = 1 to K:
3.     v_new = M · v
4.     If ||v_new - v||₁ < ε:
5.         return v_new  (converged)
6.     v = v_new
7. Return v  (reached max iterations)
```

**Why this works:** From Section 2.2.2, repeated multiplication by $M$ amplifies the component of $\mathbf{v}$ in the direction of the dominant eigenvector while suppressing all other components. The spectral decomposition shows that:

$$
M^k \mathbf{v}_0 = \lambda_1^k \left( c_1 \mathbf{q}_1 + \sum_{i=2}^{n} c_i \left(\frac{\lambda_i}{\lambda_1}\right)^k \mathbf{q}_i \right)
$$

For stochastic $M$ with $\lambda_1 = 1$ and $|\lambda_i| < 1$ for $i \geq 2$, the second term vanishes exponentially, leaving $\mathbf{v}_k \to c_1 \mathbf{q}_1$.

#### 2.3.2 The Perron-Frobenius Theorem

The convergence guarantee for Power Iteration on stochastic matrices relies on the celebrated **Perron-Frobenius theorem**. We state and prove it for our setting.

**Theorem 2.2 (Perron-Frobenius for Primitive Stochastic Matrices):**  
Let $M$ be an irreducible, aperiodic, column-stochastic $n \times n$ matrix. Then:

1. $\lambda = 1$ is an eigenvalue of $M$
2. $\lambda = 1$ is simple (algebraic multiplicity 1) and dominant: $|\lambda_i| < 1$ for all $i \neq 1$
3. The corresponding right eigenvector $\mathbf{r}$ can be chosen strictly positive: $r_i > 0$ for all $i$
4. This eigenvector is unique (up to scalar multiplication)

**Definitions:**
- **Irreducible:** For any states $i, j$, there exists $k$ such that $(M^k)_{ij} > 0$ (you can reach $j$ from $i$ eventually)
- **Aperiodic:** The greatest common divisor of return times to any state is 1 (no periodic cycles)
- **Primitive:** Irreducible + aperiodic ⟹ $\exists k$ such that $M^k > 0$ (all entries positive)

**Proof Sketch:**

[**Step 1: Existence of $\lambda = 1$**]  
Column-stochastic means $\sum_{i=1}^{n} M_{ij} = 1$ for all $j$. In matrix form: $\mathbf{1}^T M = \mathbf{1}^T$ where $\mathbf{1} = [1, 1, \ldots, 1]^T$. This shows $\lambda = 1$ is an eigenvalue (with left eigenvector $\mathbf{1}^T$). To find the right eigenvector, we proceed to Step 2.

[**Step 2: Positive eigenvector via fixed point**]  
Consider the probability simplex $\Delta^n = \{\mathbf{x} \in \mathbb{R}^n : x_i \geq 0, \sum_i x_i = 1\}$. Since $M$ is column-stochastic, it maps $\Delta^n$ to itself. By the Brouwer Fixed Point Theorem [15], there exists $\mathbf{r} \in \Delta^n$ such that $M\mathbf{r} = \mathbf{r}$. This is our eigenvector with $\lambda = 1$, and $\mathbf{r} \geq 0$ by construction.

[**Step 3: Simplicity and uniqueness**]  
For a primitive matrix (irreducible + aperiodic), there exists $k$ such that $M^k$ has all strictly positive entries [16, Theorem 8.5.1]. This property implies the Perron root $\lambda = 1$ has algebraic multiplicity 1, and the eigenvector is unique up to scaling.

[**Step 4: Dominance of $\lambda = 1$**]  
Primitivity also ensures that all other eigenvalues satisfy $|\lambda_i| < 1$ [16, Theorem 8.5.2]. Intuitively: irreducibility means the chain can reach all states, and aperiodicity prevents getting stuck in cycles. Together, these force convergence to a unique steady state, which corresponds to the dominant eigenvalue being strictly larger than all others. ∎

**Remark:** The full proof of Steps 3 and 4 requires the theory of non-negative matrices and is beyond our scope. We refer to Meyer [16, Chapter 8] for complete details.

#### 2.3.3 Convergence Rate

The speed of convergence depends on the **spectral gap**.

**Definition 2.3 (Spectral Gap):**  
For a stochastic matrix $M$ with eigenvalues ordered by magnitude $|\lambda_1| \geq |\lambda_2| \geq \cdots \geq |\lambda_n|$, the spectral gap is:

$$
\delta = 1 - |\lambda_2|
$$

where $\lambda_1 = 1$ (for stochastic matrices).

**Convergence bound:** The error at iteration $k$ satisfies:

$$
\|\mathbf{v}_k - \mathbf{r}\|_1 \lesssim |\lambda_2|^k \cdot \|\mathbf{v}_0 - \mathbf{r}\|_1
$$

**Interpretation:**
- **Large gap** ($\delta$ close to 1 ⟹ $|\lambda_2|$ close to 0): Fast exponential convergence
- **Small gap** ($\delta$ close to 0 ⟹ $|\lambda_2|$ close to 1): Slow convergence

**Practical rule:** To achieve error $\epsilon$, we need approximately:

$$
k \approx \frac{\log(1/\epsilon)}{\log(1/|\lambda_2|)} = \frac{\log(1/\epsilon)}{-\log|\lambda_2|}
$$

iterations [17].

**Example:** For PageRank with damping factor $d = 0.85$, the second eigenvalue satisfies $|\lambda_2| \leq d < 1$ (proven in Section 2.4). This typically gives convergence in 30-50 iterations for $\epsilon = 10^{-6}$.

**Visualization concept:** A plot of $\|\mathbf{v}_k - \mathbf{v}_{k-1}\|_1$ vs. iteration $k$ on a logarithmic scale shows a straight line (exponential decay), with slope determined by $|\lambda_2|$. We will generate this plot in Section 3.

---

### 2.4 Handling Real-World Issues: The Damping Factor

#### 2.4.1 Problems with Naive PageRank

The basic model $M\mathbf{r} = \mathbf{r}$ assumes the web graph is well-behaved. Real web graphs exhibit pathologies that violate the conditions of Theorem 2.2:

**Problem 1: Dangling Nodes (Dead Ends)**  
Pages with no outgoing links (e.g., PDFs, images, or pages behind paywalls). The corresponding column in the adjacency matrix is all zeros, making normalization impossible:

$$
M_{ij} = \frac{A_{ij}}{\sum_k A_{kj}} = \frac{0}{0} \quad \text{(undefined)}
$$

**Consequence:** The random surfer gets "trapped"—probability accumulates at dangling nodes and never propagates back to the rest of the graph.

**Problem 2: Spider Traps (Isolated Cycles)**  
Clusters of pages that link only to each other, disconnected from the broader web. Example: two pages $A \leftrightarrow B$ with no external links.

**Consequence:** Probability "leaks" into the trap and circulates forever. The stationary distribution concentrates all mass on the trap, assigning zero importance to the rest of the web—clearly undesirable.

**Mathematical issue:** These pathologies break **irreducibility**. The graph has multiple disconnected components, so Theorem 2.2 does not apply. The dominant eigenvalue may still be $\lambda = 1$, but the eigenvector is not unique or may not be positive.

#### 2.4.2 The Teleportation Model

The **damping factor** modification solves these problems by introducing **random teleportation** [1].

**Probabilistic Model:**  
At each step, the random surfer:
- With probability $d$ (typically 0.85): Follows a random outlink from the current page
- With probability $1 - d$: "Gets bored" and jumps to a uniformly random page (teleportation)

Let $\tilde{M}$ be the normalized transition matrix (handling dangling nodes by adding uniform distribution from dead ends). The modified transition probability is:

$$
P(X_{t+1} = i \mid X_t = j) = d \cdot \tilde{M}_{ij} + (1-d) \cdot \frac{1}{n}
$$

**Matrix Formulation:**  
Define the **teleportation matrix**:

$$
E = \mathbf{1} \cdot \mathbf{1}^T = \begin{bmatrix}
1 & 1 & \cdots & 1 \\
1 & 1 & \cdots & 1 \\
\vdots & \vdots & \ddots & \vdots \\
1 & 1 & \cdots & 1
\end{bmatrix} \in \mathbb{R}^{n \times n}
$$

where $E\mathbf{v} = \|\mathbf{v}\|_1 \cdot \mathbf{1}$ (sums all entries and distributes uniformly). The **modified PageRank matrix** is:

$$
M' = d \cdot \tilde{M} + \frac{(1-d)}{n} \cdot E
$$

**Verification that $M'$ is stochastic:**

$$
\sum_{i=1}^{n} M'_{ij} = d \sum_{i=1}^{n} \tilde{M}_{ij} + \frac{(1-d)}{n} \sum_{i=1}^{n} E_{ij} = d \cdot 1 + \frac{(1-d)}{n} \cdot n = d + (1-d) = 1 \quad \checkmark
$$

#### 2.4.3 Why Damping Ensures Convergence

**Theorem 2.3:**  
For any $d \in [0, 1)$, the matrix $M'$ is **primitive** (irreducible and aperiodic), ensuring the Perron-Frobenius conditions hold.

**Proof Sketch:**

- **Irreducibility:** The term $\frac{(1-d)}{n} E$ adds a positive probability of transitioning from any state to any other state. Thus, $(M')_{ij} > 0$ for all $i, j$, making $M'$ **fully connected**.
  
- **Aperiodicity:** Since $(M')_{ii} > 0$ for all $i$ (non-zero diagonal entries from teleportation), the chain can stay in any state, breaking all cycles.

Therefore, $M'$ is primitive, and Theorem 2.2 guarantees a unique positive stationary distribution $\mathbf{r}$ with $M'\mathbf{r} = \mathbf{r}$. ∎

**Additional benefit:** The damping factor also controls the second eigenvalue. It can be shown that $|\lambda_2| \leq d$ for $M'$ [18], so increasing $d$ (more faithful link-following) slows convergence, while decreasing $d$ (more random jumping) speeds it up.

---

> **Why $d = 0.85$?**  
> The choice $d = 0.85$ originated in the original PageRank paper [1] and represents a **pragmatic balance**:
> - **Too high** ($d \to 1$): Slow convergence and vulnerability to spider traps
> - **Too low** ($d \to 0$): Fast convergence but rankings become uniform (all pages equally important)
> - **$d = 0.85$**: Empirically balances accuracy and efficiency. Interpreted as: a random surfer follows ~5-6 links before getting bored and starting fresh elsewhere (since $(0.85)^5 \approx 0.44$ and $(0.85)^6 \approx 0.38$).

---

### 2.5 Eigenvector Formulation and Steady-State

#### 2.5.1 The PageRank Equation

Combining our derivations, the **PageRank vector** $\mathbf{r}$ is the solution to:

$$
\mathbf{r} = M' \mathbf{r}
$$

subject to normalization $\|\mathbf{r}\|_1 = 1$ and non-negativity $r_i \geq 0$.

This is **precisely the eigenvector equation**:

$$
M' \mathbf{r} = \lambda \mathbf{r} \quad \text{with } \lambda = 1
$$

**Why $\lambda = 1$?**  
Since $M'$ is column-stochastic, multiplying any probability distribution $\mathbf{v}$ (with $\|\mathbf{v}\|_1 = 1$) by $M'$ yields another probability distribution:

$$
\|M'\mathbf{v}\|_1 = \sum_{i=1}^{n} (M'\mathbf{v})_i = \sum_{i=1}^{n} \sum_{j=1}^{n} M'_{ij} v_j = \sum_{j=1}^{n} v_j \underbrace{\sum_{i=1}^{n} M'_{ij}}_{=1} = \sum_{j=1}^{n} v_j = 1
$$

Thus, the transformation preserves total probability, corresponding to eigenvalue $\lambda = 1$.

#### 2.5.2 Operator Norms and Spectral Theory

For completeness, we introduce the norms used in convergence analysis.

**Definition 2.4 (Vector Norms):**
- **$L_1$ norm (Manhattan):** $\|\mathbf{v}\|_1 = \sum_{i=1}^{n} |v_i|$
- **$L_2$ norm (Euclidean):** $\|\mathbf{v}\|_2 = \sqrt{\sum_{i=1}^{n} v_i^2}$
- **$L_\infty$ norm (Max):** $\|\mathbf{v}\|_\infty = \max_{i} |v_i|$

**Definition 2.5 (Induced Matrix Norms):**  
For a matrix $A \in \mathbb{R}^{n \times n}$, the **operator norm** induced by vector norm $\|\cdot\|_p$ is:

$$
\|A\|_p = \sup_{\mathbf{v} \neq \mathbf{0}} \frac{\|A\mathbf{v}\|_p}{\|\mathbf{v}\|_p} = \sup_{\|\mathbf{v}\|_p = 1} \|A\mathbf{v}\|_p
$$

**Key properties for stochastic matrices:**
- $\|M'\|_1 = \max_j \sum_{i=1}^{n} |M'_{ij}| = 1$ (maximum column sum)
- The **spectral norm** $\|M'\|_2 = \sqrt{\rho(M'^T M')}$ (largest singular value), where $\rho(A)$ denotes the **spectral radius** (maximum eigenvalue magnitude)

**Spectral Theory Connection:**  
The study of eigenvalues and their geometric/dynamic implications is called **spectral theory** [19]. For PageRank:
- The **dominant eigenvalue** $\lambda_1 = 1$ determines the steady-state
- The **spectral gap** $1 - |\lambda_2|$ controls convergence speed
- The **spectral radius** $\rho(M') = 1$ reflects that $M'$ is norm-preserving on probability distributions

These concepts unify our understanding: finding PageRank reduces to computing the dominant eigenvector, and the convergence behavior is governed by the eigenvalue spectrum.

#### 2.5.3 Connection to Numerical Validation

In Section 3 (Implementation), we will:
1. **Build** $M'$ from a web graph using NumPy arrays
2. **Iterate** $\mathbf{v}_{k+1} = M' \mathbf{v}_k$ until convergence (Power Iteration)
3. **Validate** by computing eigenvalues/eigenvectors analytically via `numpy.linalg.eig()`

The validation step verifies that our iterative approximation $\mathbf{v}_k$ matches the true eigenvector from linear algebra. If $\|\mathbf{v}_k - \mathbf{r}_{\text{eig}}\|_1 < 10^{-3}$, we confirm:
- Our Power Iteration implementation is correct
- Our stopping criterion $\epsilon = 10^{-6}$ is appropriate
- The algorithm converged to the mathematical steady-state

This dual approach (iterative + analytical) demonstrates both computational efficiency and mathematical rigor.

---

> ⚠️ **Common Misconception:**  
> Students sometimes think PageRank simply counts the number of inlinks—"the page with the most links wins." This is **incorrect**. PageRank finds the page that is linked to by **important pages**, not just many pages. A page with 1,000 spam links may have lower PageRank than a page with 5 links from highly authoritative sources like `.edu` or `.gov` domains. The recursive nature ($\mathbf{r} = M'\mathbf{r}$) is what distinguishes PageRank from naive link counting.

---

### Summary of Section 2

We have established the mathematical foundation for PageRank:

1. **Markov Chain Model (2.1):** Web navigation as a memoryless random walk on a finite state space, simplifying to discrete probability
2. **Spectral Decomposition (2.2):** Repeated matrix multiplication amplifies the dominant eigenvector, explaining convergence
3. **Power Iteration (2.3):** An efficient iterative algorithm guaranteed to converge by the Perron-Frobenius theorem
4. **Damping Factor (2.4):** Teleportation fixes pathologies (dangling nodes, spider traps) by ensuring irreducibility and aperiodicity
5. **Eigenvector Equation (2.5):** PageRank is the eigenvector of $M'$ with eigenvalue $\lambda = 1$, computable numerically and verifiable analytically

In Section 3, we translate this theory into Python code, implementing the algorithm on a mini-web and validating against eigenvalue decomposition.

---

### References (Section 2)

[10] Norris, J. R. (1998). *Markov Chains*. Cambridge University Press. Chapter 1: Discrete-Time Markov Chains.

[11] Durrett, R. (2019). *Probability: Theory and Examples* (5th ed.). Cambridge University Press. Chapter 6: Markov Chains (measure-theoretic treatment).

[12] Ross, S. M. (2014). *Introduction to Probability Models* (11th ed.). Academic Press. Chapter 4: Markov Chains (elementary approach).

[13] Strang, G. (2016). *Introduction to Linear Algebra* (5th ed.). Wellesley-Cambridge Press. Chapter 6: Eigenvalues and Eigenvectors.

[14] Golub, G. H., & Van Loan, C. F. (2013). *Matrix Computations* (4th ed.). Johns Hopkins University Press. Section 7.3: Power Iteration.

[15] Brouwer, L. E. J. (1911). Über Abbildung von Mannigfaltigkeiten. *Mathematische Annalen*, 71(1), 97-115.

[16] Meyer, C. D. (2000). *Matrix Analysis and Applied Linear Algebra*. SIAM. Chapter 8: The Perron-Frobenius Theory of Nonnegative Matrices.

[17] Langville, A. N., & Meyer, C. D. (2006). *Google's PageRank and Beyond: The Science of Search Engine Rankings*. Princeton University Press. Chapter 3: Convergence Analysis.

[18] Haveliwala, T. H., & Kamvar, S. D. (2003). *The Second Eigenvalue of the Google Matrix*. Stanford University Technical Report. http://ilpubs.stanford.edu:8090/630/

[19] Horn, R. A., & Johnson, C. R. (2013). *Matrix Analysis* (2nd ed.). Cambridge University Press. Chapter 5: Spectral Theory.

---
                                                                                                                                                                                                    